<a href="https://colab.research.google.com/github/firstaidspray/marvel-rivals-cosmetics-ab-test/blob/main/Marvel_Rivals_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This pipeline uses Master Data Management (MDM) to join raw tracker data against the canonical Fandom Wiki taxonomy..."

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# 1. EXTRACT: Fetch the data
def get_raw_html():
    url = "https://rivalsmeta.com/skins"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    print("Attempting to fetch live data from RivalsMeta API...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            print("Success! Live data extracted.")
            return response.text
        else:
            print(f"Live request blocked by site security (Status: {response.status_code}).")
    except Exception as e:
        print(f"Connection failed: {e}")

    print("\n--- INITIATING MANUAL OVERRIDE ---")
    print("To bypass security, go to https://rivalsmeta.com/skins in your browser.")
    print("Press Ctrl+S (or Cmd+S) to save the webpage as 'rivalsmeta.html'.")
    print("Upload 'rivalsmeta.html' to this Colab notebook's file folder.")

    try:
        with open("rivalsmeta.html", "r", encoding="utf-8") as f:
            print("\nLocal HTML file found. Extracting data...")
            return f.read()
    except FileNotFoundError:
        print("Waiting for 'rivalsmeta.html' to be uploaded...")
        return None

html_data = get_raw_html()

if html_data:
    # 2. TRANSFORM: Parse the HTML and enrich with the Source of Truth
    soup = BeautifulSoup(html_data, 'html.parser')

    # Extract all text blocks that contain percentages
    extracted_data = []

    # NEW REGEX LINE: Added the period inside the character bracket [A-Za-z0-9\s\-\'?!.]
    text_content = soup.get_text(separator=' | ')
    matches = re.findall(r'([A-Za-z0-9\s\-\'?!.]+)\s*\|\s*([0-9]+\.[0-9]+)%', text_content)

    for match in matches:
        skin_name = match[0].strip()
        usage_pct = float(match[1])
        extracted_data.append({"Skin_Name": skin_name, "Usage_Percent": usage_pct})

    df = pd.DataFrame(extracted_data)

    # Remove duplicates if any exist from the HTML layout
    df = df.drop_duplicates(subset=['Skin_Name'])

    # Source of Truth from the Wiki Document
    lucky_draw_skins = [
        "Shadowed Start", "Maid-Pool", "Nurse-Pool", "Jeff the Dinosaur",
        "Nautical Mystic", "White Wave Queen", "Mr. X", "Unseen Weaver",
        "Vampy Jammies", "Seaside Siren", "Lunar Luna", "Soul Surfer",
        "Master of Slumber", "Mr. Malignance", "Cosmic Spark", "Daring Daifuku",
        "Mrs. X", "Seaside Sorceress", "Coastal Kumiho"
    ]

    # Standardize casing for accurate matching
    df['Skin_Name_Clean'] = df['Skin_Name'].str.title()
    lucky_draw_clean = [skin.title() for skin in lucky_draw_skins]

    df['Monetization_Type'] = df['Skin_Name_Clean'].apply(
        lambda x: 'Gacha (Lucky Draw)' if x in lucky_draw_clean else 'Store / Earned'
    )

    # 3. LOAD & ANALYZE
    print("\n=== MARVEL RIVALS: MONETIZATION A/B TEST ===")

    ab_test_results = df.groupby('Monetization_Type').agg(
        Skin_Count=('Skin_Name', 'count'),
        Average_Usage_Percent=('Usage_Percent', 'mean'),
        Max_Usage_Percent=('Usage_Percent', 'max')
    ).round(2)

    print(ab_test_results)

    df.to_csv("marvel_rivals_ab_test_clean.csv", index=False)
    print("\nClean dataset 'marvel_rivals_ab_test_clean.csv' is ready for Tableau!")

Attempting to fetch live data from RivalsMeta API...
Success! Live data extracted.

=== MARVEL RIVALS: MONETIZATION A/B TEST ===
                    Skin_Count  Average_Usage_Percent  Max_Usage_Percent
Monetization_Type                                                       
Gacha (Lucky Draw)          19                  19.31              48.20
Store / Earned             654                   7.42              75.34

Clean dataset 'marvel_rivals_ab_test_clean.csv' is ready for Tableau!


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# 1. EXTRACT: Fetch the data
def get_raw_html():
    url = "https://rivalsmeta.com/skins"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    print("Attempting to fetch live data from RivalsMeta API...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            print("Success! Live data extracted.")
            return response.text
        else:
            print(f"Live request blocked by site security (Status: {response.status_code}).")
    except Exception as e:
        print(f"Connection failed: {e}")

    print("\n--- INITIATING MANUAL OVERRIDE ---")
    print("To bypass security, go to https://rivalsmeta.com/skins in your browser.")
    print("Press Ctrl+S (or Cmd+S) to save the webpage as 'rivalsmeta.html'.")
    print("Upload 'rivalsmeta.html' to this Colab notebook's file folder.")

    try:
        with open("rivalsmeta.html", "r", encoding="utf-8") as f:
            print("\nLocal HTML file found. Extracting data...")
            return f.read()
    except FileNotFoundError:
        print("Waiting for 'rivalsmeta.html' to be uploaded...")
        return None

html_data = get_raw_html()

if html_data:
    # 2. TRANSFORM: Parse the HTML text positionally
    soup = BeautifulSoup(html_data, 'html.parser')
    text_content = soup.get_text(separator=' | ')

    # Step A: Map out where every Hero section starts in the text
    # The site headers look like "Invisible Woman Skins | 27 skins"
    hero_pattern = r'([A-Za-z0-9\s\-\'.]+)\s+Skins\s*\|\s*[0-9]+\s*skins'
    hero_positions = []

    for match in re.finditer(hero_pattern, text_content, re.IGNORECASE):
        hero_name = match.group(1).strip()
        # Clean up any trailing text before the word "Skins"
        hero_name = hero_name.split('|')[-1].strip()
        pos = match.start()
        hero_positions.append({"hero": hero_name, "pos": pos})

    # Step B: Uses \D* to blindly skip all invisible text/characters until it hits the match count
    skin_pattern = r'([A-Za-z0-9\s\-\'?!.]+)\s*\|\s*([0-9]+\.[0-9]+)%\D*([0-9,]+)\s*matches'
    extracted_data = []

    for match in re.finditer(skin_pattern, text_content, re.IGNORECASE):
        skin_name = match.group(1).strip()
        usage_pct = float(match.group(2))

        # Grab group 3, remove the comma, and turn it into a pure integer
        matches_seen = int(match.group(3).replace(',', ''))

        pos = match.start()

        # Find which hero section we are currently in based on character position
        current_hero = "Unknown"
        past_heroes = [h for h in hero_positions if h["pos"] < pos]
        if past_heroes:
            current_hero = past_heroes[-1]["hero"]

        extracted_data.append({
            "Hero_Name": current_hero,
            "Skin_Name": skin_name,
            "Usage_Percent": usage_pct,
            "Matches_Seen": matches_seen  # New raw volume metric added
        })

    df = pd.DataFrame(extracted_data)

    # Remove duplicates if the site layout repeats skins
    df = df.drop_duplicates(subset=['Skin_Name'])

    # Step C: Apply the Business Logic (Source of Truth)
    lucky_draw_skins = [
        "Shadowed Start", "Maid-Pool", "Nurse-Pool", "Jeff the Dinosaur",
        "Nautical Mystic", "White Wave Queen", "Mr. X", "Unseen Weaver",
        "Vampy Jammies", "Seaside Siren", "Lunar Luna", "Soul Surfer",
        "Master of Slumber", "Mr. Malignance", "Cosmic Spark", "Daring Daifuku",
        "Mrs. X", "Seaside Sorceress", "Coastal Kumiho"
    ]

    df['Skin_Name_Clean'] = df['Skin_Name'].str.title()
    lucky_draw_clean = [skin.title() for skin in lucky_draw_skins]

    df['Monetization_Type'] = df['Skin_Name_Clean'].apply(
        lambda x: 'Gacha (Lucky Draw)' if x in lucky_draw_clean else 'Store / Earned'
    )

    # Reorder columns to include Matches_Seen for a clean upload
    df = df[['Hero_Name', 'Skin_Name', 'Usage_Percent', 'Matches_Seen', 'Monetization_Type']]

    # 3. LOAD & ANALYZE
    print(f"\nSuccessfully extracted {len(df)} skins with dynamic Hero mapping and raw match volume!")
    print("\n--- Final Dataset Preview ---")
    print(df.head(10))

    df.to_csv("marvel_rivals_ab_test_final.csv", index=False)
    print("\nFile 'marvel_rivals_ab_test_final.csv' is generated and ready for Google Sheets!")

Attempting to fetch live data from RivalsMeta API...
Success! Live data extracted.

Successfully extracted 673 skins with dynamic Hero mapping and raw match volume!

--- Final Dataset Preview ---
         Hero_Name               Skin_Name  Usage_Percent  Matches_Seen  \
0  Invisible Woman           UNSEEN WEAVER          21.53        150301   
1  Invisible Woman                  Malice          10.13         70760   
2  Invisible Woman      The Life Fantastic           6.99         48817   
3  Invisible Woman            Blood Shield           6.90         48147   
4  Invisible Woman             AZURE SHADE           6.81         47518   
5  Invisible Woman             First Steps           5.59         39043   
6  Invisible Woman               RUBY RITE           5.43         37883   
7  Invisible Woman          Lush Luminance           4.51         31489   
8  Invisible Woman            PRISM PARADE           3.48         24331   
9  Invisible Woman  BAXTER BUILDING BEAUTY           2